# Bangla RAG evaluation: Colab runner
Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Add your key under the key icon (Secrets) as `GEMINI_API_KEY` and switch on notebook access.

In [ ]:
# 1. Get the code. Option A: clone your GitHub repo (edit the URL). Option B: upload the project zip and unzip it.
REPO_URL = "https://github.com/YOUR_USERNAME/bangla-rag-evaluation.git"
!git clone $REPO_URL repo
%cd repo
!pip -q install -r requirements.txt

In [ ]:
# 2. API key from Colab Secrets (never paste the key into a cell)
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
!python -m pytest -q

In [ ]:
# 3. Download BanglaRQA
!python -m bangla_rag.data

In [ ]:
# 4. Retrieval-only metrics: free, no Gemini calls. Checks the dense models and FAISS work.
!python -m bangla_rag.run --preset core --retrieval-only --n 0 --out results

In [ ]:
# 5. Smoke test: 5 questions through Gemini, separate output folder
!python -m bangla_rag.run --runs none,hybrid-bge-m3_c512_k3 --n 5 --rpm 10 --out results_smoke
import json
for line in open('results_smoke/hybrid-bge-m3_c512_k3.jsonl', encoding='utf-8'):
    r = json.loads(line); print(r['question'], '->', r['answer'], '| refs:', r['references'], '| F1', round(r['f1'], 2))

In [ ]:
# 6. Full core run (resumable; if the daily free quota runs out, run this cell again later)
!python -m bangla_rag.run --preset core --n 200 --rpm 10 --out results
!python -m bangla_rag.analyze --results results

In [ ]:
# 7. Optional: the full ablation (adds embedders, chunk sizes, top-5)
!python -m bangla_rag.run --preset full --n 200 --rpm 10 --out results
!python -m bangla_rag.analyze --results results

In [ ]:
# 8. RQ2: create the human grading sheet, then download it and grade it by hand
!python -m bangla_rag.human sample --runs bm25_c512_k3,hybrid-bge-m3_c512_k3 --n 100
from google.colab import files
files.download('human/grading_sheet.csv')

In [ ]:
# 9. After grading: upload the filled sheet to human/grading_sheet.csv, then run RAGAS and compare
!pip -q install -r requirements-ragas.txt
!python -m bangla_rag.ragas_eval --tag en
!python -m bangla_rag.human analyze --ragas human/ragas_en.csv

In [ ]:
# 10. Save results to your Google Drive so they survive the session
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bangla_rag_results
!cp -r results human /content/drive/MyDrive/bangla_rag_results/